In [20]:
import yfinance as yf
from stocktrends import Renko

In [21]:
tickers = ["AMZN"]
ohlcv_data = {}
hour_data = {}
renko_data = {}
for ticker in tickers:
    data = yf.download(ticker, period='1mo', interval='5m')
    data.dropna(how='any', inplace=True)
    data.columns = data.columns.droplevel(1)
    ohlcv_data[ticker] = data
    hour_data_period = yf.download(ticker, period='1mo', interval='1h')
    data.dropna(how='any', inplace=True)
    hour_data[ticker] = hour_data_period

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


In [25]:
def ATR(DF, n=14):
    df = DF.copy()
    df["H-L"] = df["High"] - df["Low"]
    df["H-PC"] = df["High"] - df["Close"].shift(1)
    df["L-PC"] = df["Low"] - df["Close"].shift(1)
    df["TR"] = df[["H-L", "H-PC","L-PC" ]].max(axis=1, skipna=False)
    df["ATR"] = df["TR"].ewm(com=n, min_periods=n).mean()
    return df["ATR"]

def renko_DF(DF, hourly_df):
    df = DF.copy()
    df.reset_index(inplace=True)
    df.columns = ["date", "open", "high", "low","close", "volume"]
    df2 = Renko(df)
    df2.brick_size = 3* round(ATR(hourly_df, 120).iloc[-1], 0)
    renko_df = df2.get_ohlc_data()
    return renko_df

In [26]:
for ticker in ohlcv_data:
    renko_data[ticker] = renko_DF(ohlcv_data[ticker], hour_data[ticker])

In [29]:
renko_data["AMZN"]

,date,open,high,low,close,uptrend
0,2026-06-15 09:30:00-04:00,234.0,240.0,234.0,240.0,True
1,2026-06-15 09:35:00-04:00,240.0,246.0,240.0,246.0,True
2,2026-06-22 11:10:00-04:00,240.0,240.0,234.0,234.0,False
3,2026-06-25 09:35:00-04:00,234.0,234.0,228.0,228.0,False
4,2026-06-29 09:50:00-04:00,234.0,240.0,234.0,240.0,True
5,2026-06-29 10:55:00-04:00,240.0,246.0,240.0,246.0,True
